<a href="https://colab.research.google.com/github/sriharikrishna/JDEV-Tutorial/blob/python/05Reverse/05Reverse.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Multiple Inputs, Multiple Outputs (MIMO)

This example computes affinity and similarity matrices from point coordinates and parameters. The function maps multiple inputs to multiple outputs.

In [ ]:
import jax
import jax.numpy as jnp

def norm_pts_jax(pt1_coords, pt2_coords):
    """Calculates the squared Euclidean distance between two points."""
    return jnp.sum((pt1_coords - pt2_coords)**2, axis=-1)

In [ ]:
def compute_matrices(nb_pts, sigma, max_val, pts_coords, connections_indices):
    """
    Computes affinity and similarity matrices, and updates point affinities.

    Returns:
        total_affinity: scalar sum of squared final affinities
        A: (nb_pts, nb_pts) normalized distance matrix
        S: (nb_pts, nb_pts) similarity matrix
    """
    diff = pts_coords[:, None, :] - pts_coords[None, :, :]
    dist_sq_matrix = jnp.sum(diff**2, axis=-1)

    A = dist_sq_matrix / (max_val * max_val)
    S = jnp.exp(-A / sigma)

    initial_affinities = jnp.ones(nb_pts, dtype=jnp.float32)

    def update_for_connection(carry, connection_pair):
        current_affinities, current_nb_neighbours = carry
        idx1, idx2 = connection_pair
        similarity_val = S[idx1, idx2]

        current_affinities = current_affinities.at[idx1].multiply(similarity_val)
        current_affinities = current_affinities.at[idx2].multiply(similarity_val)
        current_nb_neighbours = current_nb_neighbours.at[idx1].add(1)
        current_nb_neighbours = current_nb_neighbours.at[idx2].add(1)

        return (current_affinities, current_nb_neighbours), None

    (final_affinities, _), _ = jax.lax.scan(
        update_for_connection,
        (initial_affinities, jnp.zeros(nb_pts, dtype=jnp.int32)),
        connections_indices
    )

    total_affinity = jnp.sum(final_affinities**2)
    return total_affinity, A, S

In [ ]:
nb_pts = 5
sigma = 20.0
max_val = 10.0

pts_coords = jnp.array([
    [1.0, 1.0, 1.0],
    [0.0, 1.0, 1.0],
    [0.0, 0.0, 1.0],
    [-1.0, -1.0, 1.0],
    [-1.0, -1.0, -1.0]
], dtype=jnp.float32)

connections = []
for i in range(nb_pts):
    for j in range(i + 1, nb_pts, i + 1 if i > 0 else 1):
        connections.append([i, j])
connections_indices = jnp.array(connections, dtype=jnp.int32)

print(f"sigma = {sigma} \t -- \t max_val = {max_val}")
print("Connections:")
for conn_idx in connections_indices:
    print(f"  ({conn_idx[0]}, {conn_idx[1]})")

total_affinity, A, S = compute_matrices(
    nb_pts, sigma, max_val, pts_coords, connections_indices
)
print(f"Total affinity = {total_affinity}")

### Reverse-mode derivatives with `jax.vjp`

1. [`jax.vjp`](https://jax.readthedocs.io/en/latest/jax.html#jax.vjp)
2. Uses reverse mode and returns a function (`vjp_fun` below) that computes $v \cdot J$ for a function $R^m \rightarrow R^n$.
3. `jax.vjp()` requires the input value for the primal function, which is the point where the derivatives are computed.
4. `vjp_fun` requires a seed $v$. For reverse mode the shape of the seed must match the primal output.
5. Exercise: see how values change as the seed changes.

`compute_matrices` returns three outputs `(total_affinity, A, S)`, so the cotangent seed is a tuple of the same structure. To get $\partial(\text{total\_affinity})/\partial \sigma$ and $\partial(\text{total\_affinity})/\partial \text{max\_val}$, use the seed `(1.0, 0, 0)` on the outputs.

In [ ]:
def compute_from_params(sigma, max_val):
    return compute_matrices(nb_pts, sigma, max_val, pts_coords, connections_indices)

primals, vjp_fun = jax.vjp(compute_from_params, sigma, max_val)
aff, A_primal, S_primal = primals

seed_aff = (1.0, jnp.zeros_like(A_primal), jnp.zeros_like(S_primal))
daff_dsigma, daff_dmax = vjp_fun(seed_aff)

print(f"Primal total affinity = {aff}")
print(f"d(total_affinity)/d(sigma) = {daff_dsigma}")
print(f"d(total_affinity)/d(max_val) = {daff_dmax}")
print(f"d(A)/d(sigma) shape = {A_primal.shape}, d(S)/d(sigma) shape = {S_primal.shape}")

For a vector input such as `pts_coords`, the cotangent seed must match the output structure. Use `(1.0, 0, 0)` to differentiate `total_affinity` with respect to all coordinates, or seed individual entries of `A` or `S` to differentiate those outputs.

In [ ]:
def compute_from_coords(coords):
    return compute_matrices(nb_pts, sigma, max_val, coords, connections_indices)

primals, vjp_fun = jax.vjp(compute_from_coords, pts_coords)
aff, A_primal, S_primal = primals

seed_aff = (1.0, jnp.zeros_like(A_primal), jnp.zeros_like(S_primal))
daff_dx00 = vjp_fun(seed_aff)[0][0, 0]

seed_A = (0.0, jnp.zeros_like(A_primal).at[0, 1].set(1.0), jnp.zeros_like(S_primal))
dA_dx00 = vjp_fun(seed_A)[0][0, 0]

seed_S = (0.0, jnp.zeros_like(A_primal), jnp.zeros_like(S_primal).at[0, 1].set(1.0))
dS_dx00 = vjp_fun(seed_S)[0][0, 0]

print(f"d(total_affinity)/d(pts_coords[0,0]) = {daff_dx00}")
print(f"d(A)/d(pts_coords[0,0]) at A[0,1] = {dA_dx00}")
print(f"d(S)/d(pts_coords[0,0]) at S[0,1] = {dS_dx00}")